# VietVoice Studio
**Văn bản của bạn. Giọng nói của bạn.**

Tạo tiếng nói từ giọng mẫu của bạn **ngay trong Google Colab**.
Notebook chứa sẵn mã nguồn, không cần cài gì trên máy, không cần liên kết Gradio hay đăng nhập phụ.

1. Chọn **Runtime → Change runtime type → T4 GPU** (nếu khả dụng).
2. Chạy lần lượt **3 ô mã** bên dưới.
3. Bộ giọng có sẵn tự tải từ GitHub trong ô 2; chọn giọng ở ô 3 rồi nhập nội dung.
4. Bấm **Tạo giọng nói**, nghe trong notebook và bấm **Tải WAV**.

Cần Internet để cài thư viện/tải trọng số. Không cam kết thời gian tạo vì tùy GPU và độ dài.
Văn bản và âm thanh được tải lên và xử lý trong phiên Colab của bạn.
Tải bản đọc về trước khi xóa/ngắt phiên. Chưa có bước huấn luyện mô hình mới.
**158 mẫu**: 8 giọng ban đầu + 50 tiếng Việt + 50 thịnh hành toàn cầu + 50 dùng nhiều toàn cầu.
Bộ giọng tự tải từ kho công khai **cracksteam456/voice-studio** mỗi phiên;
không cần GitHub token hoặc tải giọng mẫu từ máy lên.
Dùng giọng của bạn hoặc giọng được chủ sở hữu cho phép sử dụng.


## 1. Cài thư viện
Chạy trên một phiên Colab mới để tránh xung đột thư viện đã nạp.


In [ ]:
REQUIREMENTS = "# GPU wheels are selected in the Colab setup cell.\ntorch==2.8.0\ntorchaudio==2.8.0\n# TorchCodec 0.7.0 is installed separately from the PyTorch CPU index in Colab.\nomnivoice==0.2.1\ntransformers==5.10.1\nsetuptools==84.2.1\ngradio==6.32.0\nnumpy==2.2.6\nsoundfile==0.13.1\n# Use Colab's preloaded ipywidgets; upgrading it mid-session breaks widget views.\n"
#@title 1. Cài thư viện (chạy một lần trên phiên mới)
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này dành cho Google Colab. Hãy mở bằng colab.research.google.com.")
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                         capture_output=True, text=True, check=False)
except FileNotFoundError:
    gpu = None
if gpu is None or gpu.returncode:
    raise RuntimeError("Hãy bật GPU: Runtime → Change runtime type → T4 GPU, rồi chạy lại.")
print("GPU:", gpu.stdout.strip())

# Cài bằng pip của đúng kernel; không thực thi script tải về từ URL.
wanted = "2.8.0"
try:
    torch_ok = (all(metadata.version(name).split("+")[0] == wanted
                    for name in ("torch", "torchaudio"))
                and metadata.version("torchvision").split("+")[0] == "0.23.0")
except metadata.PackageNotFoundError:
    torch_ok = False
if not torch_ok:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--only-binary=:all:",
        "torch==2.8.0", "torchaudio==2.8.0", "torchvision==0.23.0",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    ])
Path("vietvoice-requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-r", "vietvoice-requirements.txt"
])
# Colab's preinstalled TorchCodec may target a different PyTorch ABI.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--only-binary=:all:", "--no-deps",
    "--force-reinstall", "torchcodec==0.7.0",
    "--index-url", "https://download.pytorch.org/whl/cpu",
])
codec_probe = subprocess.run(
    [sys.executable, "-c", "from torchcodec.decoders import AudioDecoder"],
    capture_output=True, text=True,
)
if codec_probe.returncode:
    raise RuntimeError("TorchCodec chưa khớp PyTorch. Khởi động lại phiên rồi chạy lại ô 1.")
print("Đã cài xong. Nếu Colab yêu cầu khởi động lại phiên, khởi động lại rồi chạy ô 2 và 3.")


## 2. Chuẩn bị ứng dụng và tải giọng từ GitHub
Mỗi lần chạy ô này sẽ lấy bộ mẫu hiện tại từ GitHub Release và kiểm tra checksum. Chưa tải trọng số mô hình.


In [ ]:
#@title 2. Chuẩn bị VietVoice Studio và kiểm tra GPU
from pathlib import Path
import sys
import importlib
import os

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
PROJECT = Path('/content/vietvoice-studio')
PROJECT.mkdir(parents=True, exist_ok=True)
FILES = {'voice_core.py': '"""Validation and audio processing; usable without PyTorch or model weights."""\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport math\nimport re\nimport unicodedata\n\nimport numpy as np\nimport soundfile as sf\n\nMAX_TEXT = 4000\nMAX_UPLOAD = 20 * 1024 * 1024\nCHUNK_SIZE = 350\nMAX_CHUNKS = 30\n\n\n@dataclass(frozen=True)\nclass Settings:\n    speed: float = 1.0\n    steps: int = 32\n    pause: float = 0.35\n    seed: int = 42\n\n    def validate(self):\n        for name, value, low, high, integer in (\n            ("Tốc độ", self.speed, 0.7, 1.3, False),\n            ("Số bước", self.steps, 8, 64, True),\n            ("Khoảng nghỉ", self.pause, 0, 1.5, False),\n            ("Seed", self.seed, 0, 2**32 - 1, True),\n        ):\n            if (isinstance(value, bool) or not isinstance(value, (int, float))\n                    or not math.isfinite(value) or not low <= value <= high\n                    or (integer and value != int(value))):\n                raise ValueError(f"{name} phải nằm trong khoảng {low}–{high}"\n                                 + (" và là số nguyên." if integer else "."))\n\n\ndef split_text(text: str, limit: int = CHUNK_SIZE) -> list[str]:\n    if not isinstance(text, str) or not text.strip():\n        raise ValueError("Hãy nhập văn bản cần đọc.")\n    if len(text) > MAX_TEXT:\n        raise ValueError(f"Mỗi lần tối đa {MAX_TEXT:,} ký tự.")\n    if not isinstance(limit, int) or limit < 20:\n        raise ValueError("Giới hạn đoạn phải là số nguyên từ 20 trở lên.")\n    text = unicodedata.normalize("NFC", text).replace("\\r\\n", "\\n").replace("\\r", "\\n")\n    if any(unicodedata.category(c) == "Cc" and c not in "\\n\\t" for c in text):\n        raise ValueError("Văn bản có ký tự điều khiển không hợp lệ.")\n    chunks = []\n    # Split paragraphs first, then prefer punctuation or word boundaries.\n    for paragraph in re.split(r"\\n\\s*\\n", text):\n        remaining = " ".join(paragraph.split())\n        while len(remaining) > limit:\n            window = remaining[:limit + 1]\n            stops = list(re.finditer(r"[.!?;…](?:[\\"\'”’])?\\s+", window))\n            end = stops[-1].start() + len(stops[-1].group().rstrip()) if stops else 0\n            if end < limit // 3:\n                end = window.rfind(" ", 0, limit + 1)\n            if end <= 0:\n                end = limit\n            chunks.append(remaining[:end].strip())\n            remaining = remaining[end:].lstrip()\n        if remaining:\n            chunks.append(remaining)\n    if len(chunks) > MAX_CHUNKS:\n        raise ValueError(f"Tối đa {MAX_CHUNKS} đoạn mỗi lần; hãy gộp bớt dòng trống.")\n    return chunks\n\n\ndef read_reference(filename: str) -> tuple[np.ndarray, int]:\n    if not filename:\n        raise ValueError("Hãy tải lên một đoạn giọng mẫu của bạn.")\n    path = Path(filename)\n    if not path.is_file() or path.stat().st_size > MAX_UPLOAD:\n        raise ValueError("Tệp giọng mẫu không tồn tại hoặc lớn hơn 20 MB.")\n    try:\n        with sf.SoundFile(path) as source:\n            if source.format not in {"WAV", "WAVEX", "RF64", "FLAC", "OGG", "MP3"}:\n                raise ValueError("Chỉ hỗ trợ WAV, MP3, FLAC hoặc OGG.")\n            rate = source.samplerate\n            if not 8000 <= rate <= 192000 or source.channels not in (1, 2):\n                raise ValueError("Giọng mẫu cần mono/stereo, tần số 8–192 kHz.")\n            if not 2 <= len(source) / rate <= 20:\n                raise ValueError("Giọng mẫu cần dài 2–20 giây; nên dùng 3–10 giây.")\n            samples = source.read(dtype="float32", always_2d=True)\n    except (RuntimeError, OSError) as exc:\n        raise ValueError("Không đọc được âm thanh. Hãy thử xuất lại thành WAV.") from exc\n    if not np.isfinite(samples).all():\n        raise ValueError("Âm thanh chứa dữ liệu không hợp lệ.")\n    mono = samples.mean(axis=1)\n    if float(np.sqrt(np.mean(mono**2))) < 1e-5:\n        raise ValueError("Giọng mẫu quá nhỏ hoặc chỉ có im lặng.")\n    return mono[np.newaxis, :], rate\n\n\ndef join_audio(chunks: list[np.ndarray], rate: int, pause: float) -> np.ndarray:\n    if not chunks or not isinstance(rate, int) or not 8000 <= rate <= 192000:\n        raise ValueError("Không có âm thanh hợp lệ để xuất.")\n    if not math.isfinite(pause) or not 0 <= pause <= 1.5:\n        raise ValueError("Khoảng nghỉ không hợp lệ.")\n    result = []\n    gap = np.zeros(round(rate * pause), dtype=np.float32)\n    for i, chunk in enumerate(chunks):\n        audio = np.array(chunk, dtype=np.float32, copy=True)\n        if (audio.ndim != 1 or not 0 < audio.size <= rate * 120\n                or not np.isfinite(audio).all()):\n            raise ValueError("Mô hình trả về âm thanh không hợp lệ.")\n        fade = min(round(rate * 0.005), audio.size // 2)\n        if fade:\n            ramp = np.linspace(0, 1, fade, dtype=np.float32)\n            audio[:fade] *= ramp\n            audio[-fade:] *= ramp[::-1]\n        if i:\n            result.append(gap)\n        result.append(audio)\n    joined = np.concatenate(result)\n    peak = float(np.max(np.abs(joined)))\n    if peak < 1e-6:\n        raise ValueError("Mô hình chỉ tạo im lặng. Hãy thử giọng mẫu khác.")\n    if peak > 1:\n        joined *= 0.98 / peak\n    return joined\n', 'engine.py': '"""Lazy OmniVoice adapter. The UI can start without downloading any weights."""\n\nfrom pathlib import Path\nimport tempfile\nimport threading\nimport time\nimport uuid\n\nimport soundfile as sf\n\nfrom voice_core import Settings, join_audio, read_reference, split_text\n\nMODEL_ID = "k2-fsa/OmniVoice"\n\n\nclass VoiceEngine:\n    def __init__(self, device: str = "auto"):\n        if device not in {"auto", "cpu", "cuda", "mps"}:\n            raise ValueError("Thiết bị không được hỗ trợ.")\n        self.device = device\n        self.model = None\n        self._lock = threading.Lock()\n        self._output = tempfile.TemporaryDirectory(prefix="vietvoice-")\n\n    @property\n    def output_dir(self) -> str:\n        return self._output.name\n\n    def _load_model(self):\n        if self.model is not None:\n            return\n        try:\n            import torch\n            from omnivoice import OmniVoice\n        except ImportError as exc:\n            raise RuntimeError("Chưa đủ thư viện AI. Hãy cài requirements.txt và PyTorch theo README.") from exc\n        device = self.device\n        if device == "auto":\n            device = ("cuda" if torch.cuda.is_available() else\n                      "mps" if torch.backends.mps.is_available() else "cpu")\n        if device == "cuda":\n            if not torch.cuda.is_available():\n                raise RuntimeError("Không tìm thấy CUDA. Hãy bật GPU trên Colab.")\n            major, _ = torch.cuda.get_device_capability()\n            if major < 7:\n                raise RuntimeError("GPU này quá cũ. Hãy dùng Colab T4 hoặc chạy với --device cpu.")\n        self.model = OmniVoice.from_pretrained(\n            MODEL_ID,\n            device_map=device,\n            dtype=torch.float32 if device == "cpu" else torch.float16,\n            load_asr=False,\n            asr_device="cpu",\n        )\n\n    def synthesize(self, text: str, reference: str, transcript: str,\n                   settings: Settings, report=None) -> tuple[str, str]:\n        report = report or (lambda value, message: None)\n        settings.validate()\n        chunks = split_text(text)\n        if not isinstance(transcript, str) or len(transcript) > 2000:\n            raise ValueError("Lời giọng mẫu tối đa 2.000 ký tự.")\n        ref_audio = read_reference(reference)\n        with self._lock:\n            started = time.monotonic()\n            report(0, "Đang chuẩn bị mô hình…" if self.model is not None else\n                   "Đang tải mô hình; lần đầu cần tải trọng số từ Hugging Face…")\n            self._load_model()\n            # Injection of a fake model in tests exercises the full pipeline without GPU.\n            import torch\n\n            cuda_devices = ([torch.cuda.current_device()]\n                            if str(self.model.device).startswith("cuda") else [])\n            random_context = torch.random.fork_rng(devices=cuda_devices)\n            with random_context, torch.inference_mode():\n                torch.manual_seed(int(settings.seed))\n                if str(self.model.device).startswith("mps"):\n                    torch.mps.manual_seed(int(settings.seed))\n                report(0.05, "Đang phân tích giọng mẫu…")\n                prompt = self.model.create_voice_clone_prompt(\n                    ref_audio=ref_audio, ref_text=transcript.strip() or None,\n                )\n                audios = []\n                for index, chunk in enumerate(chunks):\n                    report(0.1 + 0.85 * index / len(chunks),\n                           f"Đang đọc đoạn {index + 1}/{len(chunks)}…")\n                    audios.append(self.model.generate(\n                        text=chunk, language="vi", voice_clone_prompt=prompt,\n                        speed=float(settings.speed), num_step=int(settings.steps),\n                    )[0])\n            rate = int(self.model.sampling_rate)\n            waveform = join_audio(audios, rate, float(settings.pause))\n            # Keep only this app\'s temporary exports for at most one hour.\n            for old in Path(self.output_dir).glob("vietvoice-*.wav"):\n                try:\n                    if time.time() - old.stat().st_mtime > 3600:\n                        old.unlink()\n                except OSError:\n                    pass\n            output = Path(self.output_dir) / f"vietvoice-{uuid.uuid4().hex}.wav"\n            sf.write(output, waveform, rate, subtype="PCM_16")\n            elapsed = time.monotonic() - started\n            report(1, "Hoàn tất")\n            status = (f"Đã tạo {len(waveform) / rate:.1f} giây · {len(chunks)} đoạn · "\n                      f"{rate:,} Hz · xử lý {elapsed:.1f} giây · seed {int(settings.seed)}")\n            return str(output), status\n\n    def close(self):\n        self._output.cleanup()\n', 'colab_ui.py': '"""Native notebook controls: no web server, tunnel or separate login."""\n\nfrom pathlib import Path\nimport tempfile\n\nimport ipywidgets as widgets\nfrom IPython.display import Audio, display\n\nfrom engine import VoiceEngine\nfrom reference_library import GROUPS\nfrom voice_core import MAX_TEXT, MAX_UPLOAD, Settings, split_text\n\nEXAMPLE = (\n    "Xin chào, đây là VietVoice Studio. Biến ý tưởng thành lời nói bằng chính giọng của bạn.\\n\\n"\n    "Bạn có thể dùng bản thu để thuyết minh video, kể chuyện hoặc đọc nội dung mình yêu thích."\n)\n\n\ndef build_ui(engine=None, library=None):\n    engine = engine or VoiceEngine("cuda")\n    library = library or []\n    reference_dir = tempfile.TemporaryDirectory(prefix="vietvoice-reference-")\n    last_output = {"path": None}\n    full = widgets.Layout(width="100%")\n    text = widgets.Textarea(value=EXAMPLE, description="Văn bản", placeholder="Nhập văn bản tiếng Việt…",\n                            layout=widgets.Layout(width="100%", height="180px"))\n    reference = widgets.FileUpload(accept=".wav,.mp3,.flac,.ogg", multiple=False,\n                                   description="Tải giọng mẫu", layout=widgets.Layout(width="200px"))\n    group_counts = {key: sum(entry.get("group", "preset") == key for entry in library) for key in GROUPS}\n    group = widgets.Dropdown(\n        options=[(f"Tất cả ({len(library)} giọng)", "all")] + [\n            (f"{label} ({group_counts[key]})", key) for key, label in GROUPS.items() if group_counts[key]\n        ], value="all", description="Nhóm giọng", layout=full,\n    )\n    voice = widgets.Dropdown(\n        options=[("Giọng tải lên của tôi", "")] + [(entry["name"], entry["path"]) for entry in library],\n        value=library[0]["path"] if library else "", description="Chọn giọng", layout=full,\n    )\n    preview = widgets.Output(layout=full)\n    transcript = widgets.Textarea(description="Lời mẫu", placeholder="Chép đúng lời nói trong giọng mẫu; có thể để trống.",\n                                  layout=widgets.Layout(width="100%", height="85px"))\n    speed = widgets.FloatSlider(value=1, min=0.7, max=1.3, step=0.05,\n                                description="Tốc độ", readout_format=".2f", layout=full)\n    steps = widgets.IntSlider(value=32, min=8, max=64, description="Số bước", layout=full)\n    pause = widgets.FloatSlider(value=0.35, min=0, max=1.5, step=0.05,\n                                description="Nghỉ (giây)", readout_format=".2f", layout=full)\n    seed = widgets.BoundedIntText(value=42, min=0, max=2**32-1, description="Seed")\n    options = widgets.VBox([\n        widgets.HTML("<b>Điều chỉnh bản đọc</b>"), speed, steps, pause, seed,\n    ], layout=full)\n    summary = widgets.Label()\n    status = widgets.Label("Sẵn sàng. Mô hình sẽ tải khi bạn bấm tạo lần đầu.")\n    progress = widgets.FloatProgress(value=0, min=0, max=1, description="Tiến trình", layout=full)\n    create = widgets.Button(description="Tạo giọng nói", button_style="success", icon="play")\n    download = widgets.Button(description="Tải WAV", icon="download", disabled=True)\n    output = widgets.Output(layout=full)\n\n    def update_summary(change=None):\n        try:\n            summary.value = f"{len(text.value):,}/{MAX_TEXT:,} ký tự · {len(split_text(text.value))} đoạn"\n        except ValueError as exc:\n            summary.value = str(exc)\n\n    def report(value, message):\n        progress.value = value\n        status.value = message\n\n    def change_voice(change=None):\n        preview.clear_output(wait=True)\n        selected = next((entry for entry in library if entry["path"] == voice.value), None)\n        transcript.value = selected.get("transcript", "") if selected else ""\n        reference.disabled = bool(selected)\n        if selected:\n            with preview:\n                display(Audio(filename=selected["path"]))\n\n    def filter_voices(change=None):\n        selected = [entry for entry in library\n                    if group.value == "all" or entry.get("group", "preset") == group.value]\n        previous = voice.value\n        voice.options = [("Giọng tải lên của tôi", "")] + [(entry["name"], entry["path"]) for entry in selected]\n        voice.value = previous if previous in {entry["path"] for entry in selected} else (\n            selected[0]["path"] if selected else ""\n        )\n\n    def generate(_):\n        create.disabled = True\n        download.disabled = True\n        last_output["path"] = None\n        progress.value = 0\n        progress.bar_style = "info"\n        output.clear_output(wait=True)\n        try:\n            temporary = not bool(voice.value)\n            if temporary:\n                if len(reference.value) != 1:\n                    raise ValueError("Hãy chọn giọng có sẵn hoặc tải lên đúng một tệp giọng mẫu.")\n                item = next(iter(reference.value.values())) if isinstance(reference.value, dict) else reference.value[0]\n                content = item["content"]\n                if not 0 < len(content) <= MAX_UPLOAD:\n                    raise ValueError("Giọng mẫu cần nhỏ hơn hoặc bằng 20 MB.")\n                # Never derive a filesystem path from an uploaded filename.\n                reference_path = Path(reference_dir.name) / "reference.audio"\n                reference_path.write_bytes(bytes(content))\n            else:\n                if voice.value not in {entry["path"] for entry in library}:\n                    raise ValueError("Giọng đã chọn không nằm trong danh sách.")\n                reference_path = Path(voice.value)\n            try:\n                path, message = engine.synthesize(\n                    text.value, str(reference_path), transcript.value,\n                    Settings(speed.value, steps.value, pause.value, seed.value), report=report,\n                )\n            finally:\n                if temporary:\n                    reference_path.unlink(missing_ok=True)\n            with output:\n                display(Audio(filename=path))\n                print(message)\n            last_output["path"] = path\n            download.disabled = False\n            status.value = "Hoàn tất. Nghe bên dưới hoặc bấm Tải WAV."\n            progress.bar_style = "success"\n        except ValueError as exc:\n            status.value = str(exc)\n            progress.bar_style = "danger"\n            with output:\n                print(str(exc))\n        except OSError as exc:\n            message = str(exc).lower()\n            if "torchcodec" in message or "undefined symbol" in message:\n                status.value = "Lỗi thư viện giải mã âm thanh. Chạy lại ô 1, khởi động lại phiên, rồi chạy ô 2 và 3."\n            else:\n                status.value = "Không đọc/ghi được âm thanh. Hãy thử giọng mẫu khác hoặc chạy lại ô 2."\n            progress.bar_style = "danger"\n            with output:\n                print(status.value)\n        except Exception as exc:\n            status.value = "Chưa tạo được giọng. Kiểm tra kết nối tải mô hình và bộ nhớ GPU."\n            progress.bar_style = "danger"\n            with output:\n                print(status.value, f"Loại lỗi: {type(exc).__name__}.")\n                print("Nếu hết bộ nhớ, khởi động lại phiên và thử nội dung ngắn hơn.")\n        finally:\n            create.disabled = False\n\n    def save(_):\n        path = last_output["path"]\n        if path and Path(path).is_file():\n            from google.colab import files\n            files.download(path)\n        else:\n            status.value = "Bản WAV không còn trong phiên. Hãy tạo lại."\n            download.disabled = True\n\n    text.observe(update_summary, names="value")\n    voice.observe(change_voice, names="value")\n    group.observe(filter_voices, names="value")\n    create.on_click(generate)\n    download.on_click(save)\n    update_summary()\n    change_voice()\n    panel = widgets.VBox([\n        widgets.HTML("<h2>1. Văn bản cần đọc</h2>"), text, summary,\n        widgets.HTML("<h2>2. Giọng mẫu của bạn</h2><p>Chọn một tệp WAV, MP3, FLAC hoặc OGG, "\n                     "tối đa 20 MB. Nên dùng 3–10 giây, rõ tiếng, một người nói.</p>"),\n        group, voice, preview, reference,\n        widgets.HTML("<b>Lời nói trong giọng mẫu</b><p>Điền chính xác để tránh tải thêm Whisper. "\n                     "Để trống sẽ nhận dạng trên CPU và chậm hơn.</p>"),\n        transcript, options, widgets.HBox([create, download]), progress, status,\n        widgets.HTML("<h2>3. Bản đọc của bạn</h2>"), output,\n    ], layout=full)\n    # Hold references until the displayed panel is released by the notebook.\n    panel._vietvoice_resources = (engine, reference_dir)\n    return panel\n', 'reference_library.py': '"""Fetch the public reference pack from GitHub on each Colab session."""\n\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport tempfile\nfrom urllib.error import HTTPError\nfrom urllib.request import Request, urlopen\nimport zipfile\n\nREPOSITORY = "cracksteam456/voice-studio"\nRELEASE_TAG = "vietvoice-library-158"\nPACK_NAME = "vietvoice-library-158.zip"\nMAX_PACK = 50 * 1024 * 1024\nGROUPS = {\n    "preset": "8 giọng ban đầu",\n    "vietnamese_primary_50": "50 giọng tiếng Việt",\n    "trending_global_50": "50 giọng thịnh hành toàn cầu",\n    "usage_global_50": "50 giọng dùng nhiều toàn cầu",\n}\n\n\ndef read_library(root):\n    root = Path(root).resolve()\n    catalog = json.loads((root / "catalog.json").read_text(encoding="utf-8"))\n    if not isinstance(catalog, list) or not 1 <= len(catalog) <= 200:\n        raise ValueError("Danh sách giọng mẫu không hợp lệ.")\n    seen = set()\n    for entry in catalog:\n        filename = entry.get("file", "")\n        if (not re.fullmatch(r"[a-z0-9-]+\\.(mp3|wav|flac|ogg)", filename)\n                or filename in seen or not isinstance(entry.get("name"), str)\n                or not 0 < len(entry["name"]) <= 150\n                or entry.get("group", "preset") not in GROUPS):\n            raise ValueError("Tên hoặc đường dẫn giọng mẫu không hợp lệ.")\n        seen.add(filename)\n        path = root / filename\n        if (not path.is_file() or path.is_symlink() or path.stat().st_size > 20 * 1024 * 1024\n                or hashlib.sha256(path.read_bytes()).hexdigest() != entry.get("sha256")):\n            raise ValueError("Giọng mẫu thiếu hoặc sai checksum: " + filename)\n    return [{**entry, "path": str(root / entry["file"])} for entry in catalog]\n\n\ndef unpack_library(archive, destination):\n    destination = Path(destination)\n    if destination.exists():\n        raise ValueError("Thư mục đích phải là thư mục mới.")\n    with zipfile.ZipFile(archive) as bundle:\n        infos = bundle.infolist()\n        names = [info.filename for info in infos]\n        if (not infos or len(infos) > 201 or len(set(names)) != len(names)\n                or sum(info.file_size for info in infos) > MAX_PACK):\n            raise ValueError("Gói giọng mẫu vượt giới hạn hoặc có tệp trùng.")\n        for info in infos:\n            if (not re.fullmatch(r"(?:catalog\\.json|[a-z0-9-]+\\.(?:mp3|wav|flac|ogg))", info.filename)\n                    or (info.external_attr >> 16) & 0o170000 == 0o120000):\n                raise ValueError("Đường dẫn trong gói mẫu không an toàn.")\n        bundle.extractall(destination)\n    return read_library(destination)\n\n\ndef download_library(destination):\n    url = f"https://github.com/{REPOSITORY}/releases/download/{RELEASE_TAG}/{PACK_NAME}"\n    request = Request(url, headers={"User-Agent": "VietVoice-Studio"})\n    with urlopen(request, timeout=60) as response, tempfile.TemporaryFile() as archive:\n        count = 0\n        while block := response.read(256 * 1024):\n            count += len(block)\n            if count > MAX_PACK:\n                raise ValueError("Gói tải vượt 50 MB.")\n            archive.write(block)\n        archive.seek(0)\n        return unpack_library(archive, destination)\n\n\ndef load_library():\n    """Check the current GitHub Release every time the preparation cell runs."""\n    folder = Path(tempfile.mkdtemp(prefix="vietvoice-library-"))\n    target = folder / "samples"\n    try:\n        entries = download_library(target)\n    except HTTPError as exc:\n        raise RuntimeError(f"Không tải được bộ mẫu từ GitHub (HTTP {exc.code}). Hãy chạy lại ô 2.") from None\n    print(f"Đã tải và kiểm tra {len(entries)} giọng từ {REPOSITORY}.")\n    return entries\n'}
for name, content in FILES.items():
    (PROJECT / name).write_text(content, encoding='utf-8')
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
importlib.invalidate_caches()
import torch
from omnivoice import OmniVoice
from transformers import AutoFeatureExtractor
import voice_core, engine, reference_library
for module in (voice_core, engine, reference_library):
    importlib.reload(module)
from engine import VoiceEngine
from reference_library import load_library
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Chọn Runtime → Change runtime type → T4 GPU.')
if 'voice_engine' not in globals():
    voice_engine = VoiceEngine('cuda')
voice_library = load_library()
print('GPU:', torch.cuda.get_device_name(0))
print('Sẵn sàng. Trọng số sẽ tải khi bạn tạo giọng lần đầu.')


## 3. Tạo giọng ngay tại đây
Chọn giọng có sẵn, nhập nội dung rồi bấm **Tạo giọng nói**.
Dùng **Nhóm giọng** để lọc đủ 158 mẫu.
Hoặc chọn **Giọng tải lên của tôi** để dùng mẫu riêng **2–20 giây**.
Bạn có thể sửa nội dung và bấm tạo nhiều lần mà không cần chạy lại ô cài đặt.
**16 bước** để thử nhanh; **32 bước** là mặc định. Tối đa **4.000 ký tự/lần**.
Lần tạo đầu tiên sẽ tải trọng số từ Hugging Face.


In [ ]:
#@title 3. Mở giao diện VietVoice Studio
from google.colab import output
from IPython.display import display
import importlib
import colab_ui
importlib.reload(colab_ui)
from colab_ui import build_ui

output.enable_custom_widget_manager()
if 'panel' in globals():
    panel.close()
panel = build_ui(voice_engine, voice_library)
display(panel)


## Mẹo và xử lý sự cố
- **Hết bộ nhớ GPU:** giảm số bước xuống 16, dùng mẫu ngắn, chia nội dung; khởi động lại phiên nếu cần.
- **Tải mô hình lâu:** lần đầu cần tải trọng số; xem thông báo tiến trình ở ô 3.
- **Giọng chưa giống:** dùng mẫu rõ, một người nói, không nhạc nền; chép lời mẫu thật chính xác.
- **Số và chữ viết tắt đọc chưa đúng:** viết thành cách đọc mong muốn trước khi tạo.
- **Lời mẫu để trống:** ứng dụng tải thêm Whisper để nhận dạng trên CPU; sẽ chậm hơn.
- **Giao diện không hiện:** chạy lại ô 3 sau khi ô 1 và 2 đã thành công.
- **Xóa dữ liệu:** tải WAV về rồi chọn **Runtime → Disconnect and delete runtime**.
  Không chia sẻ notebook kèm đầu ra hoặc giọng mẫu nếu có thông tin riêng tư.
- **Đổi mã nguồn:** sửa các tệp Python rồi chạy `python scripts/build_notebook.py` để dựng lại notebook.

Tham khảo notebook của [doanquangkien/voice-notebooks](https://github.com/doanquangkien/voice-notebooks).
Mã ứng dụng được viết riêng, không kèm giọng thu của tác giả nguồn.
Dùng thư viện [k2-fsa/OmniVoice](https://github.com/k2-fsa/OmniVoice), Apache-2.0;
mô hình và các phụ thuộc giữ giấy phép riêng, xem tài liệu upstream.
